In [37]:
import glob
import os
import h5py
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 400 # User can set this outside the class if needed

from astropy.io import fits
from astropy.io.votable import parse_single_table
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.coordinates import SkyCoord
import astropy.units as u

import os
import h5py
from tqdm import tqdm
from multiprocessing import Pool 
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor
import numpy as np

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
import astropy.units as u
from astropy.table import Table

from reproject import reproject_interp
from reproject import reproject_exact
from reproject import reproject_adaptive

from scipy.sparse import coo_matrix
from scipy.sparse.linalg import lsqr
import sys 
import gc 
from functools import partial

from SelfCal.MapHelper import bit_to_bool, make_weight, find_outliers, map_pixels, det_to_grid
from SelfCal.WCSHelper import load_from_fits, save_to_fits, find_optimal_frame
from SelfCal import MakeMap
from SelfCal.SPHERExUtility import make_chunk_map, make_chunk_mask
import importlib
importlib.reload(MakeMap)

from datetime import datetime
import pandas as pd

### Inspect Exposures

In [50]:
def plot_map(map, wcs=None, lowp=0.1, highp=95, fig=None, ax=None, colorbar=True, low=None, high=None):
    if low is None or high is None:
        high, low = np.nanpercentile(map[map>0], [highp, lowp])
    if fig is None:
        fig = plt.figure(figsize=(10, 10))
    if ax is None:
        ax = fig.add_subplot(111, projection=wcs)
    im = ax.imshow(map, norm=LogNorm(vmin=low, vmax=high), origin='lower')

    if wcs is not None:
        # Explicitly set axis labels
        ax.coords['ra'].set_axislabel('RA')
        ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
        ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
        ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
        ax.coords['dec'].set_axislabel('DEC')
        ax.grid(color='black', linestyle='--', alpha=0.5)

    if colorbar:
        cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
        cbar.set_label('MJy/sr')
    return fig, ax

In [ ]:
detector = 1
exposure_list = glob.glob(f'/data1/SPHEREx/deep_north/*/*/*/*D{detector}*.fits')
hdul = fits.open(exposure_list[0])
data = hdul[1].data
wcs = WCS(hdul[1].header, naxis=2)

In [ ]:
# mask = bit_to_bool(hdul[2].data, ignore_list=[17, 21], bitmask_header=hdul[2].header, invert=True)
# masked_data = np.where(mask, data, np.nan)
# plt.imshow(masked_data, norm=LogNorm(vmin=0.1,vmax=0.2))

In [ ]:
chunk_map = make_chunk_map(detector, interp_factor=5)
chunk_valid_mask = make_chunk_mask([1], interp_factor=5)

In [ ]:
# # Do plot_map(data) and imshow(chunk_map) side by side
# fig, axes = plt.subplots(1, 2, figsize=(15, 7))
# axes[0].imshow(chunk_map, origin='lower', cmap='gray', alpha=0.5)
# plot_map(data, lowp=5, highp=95, ax=axes[1], colorbar=False)


### Inspect Reprojected Frames

In [ ]:
reproj_sample = MakeMap.load_reproj_file('/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/reprojected/exp_2195_det_00.h5', 
                         fields=['sub_data', 'file_path'])

In [ ]:
file_path = '/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/reprojected/exp_2195_det_00.h5'

In [ ]:
fields=['sub_data', 'file_path']
with h5py.File(file_path, 'r') as file:
    for key in fields:
        if key in ('sub_wcs', 'det_wcs'):
            header_key = 'sub_header' if key == 'sub_wcs' else 'det_header'
            header_str = file[header_key][()].decode('utf-8')
            data[key] = WCS(fits.Header.fromstring(header_str))
        else:
            data[key] = file[key][()]  # Efficient read

In [ ]:
# hdul = fits.open(reproj_sample['file_path'].decode('ascii'))
# data = hdul[1].data
# plot_map(data, lowp=5, highp=95, colorbar=False)

In [ ]:
# plot_map(reproj_sample['sub_data'], lowp=5, highp=95, colorbar=False)

In [ ]:
coords, data, weight, _  = MakeMap._prep_subframe(
    '/home/thomasli/spherex/selfcal/outputs/nep_det2_6p2arcsec/reprojected/exp_0000_det_00.h5', None, None, 0, 0, 
                False, True, chunk_map, chunk_valid_mask)

In [ ]:
# plot_map(data*(weight>0), lowp=5, highp=95, colorbar=False)

### Plot Mosaic

In [42]:
detector = 6
channel = 7
# fit_hdul = fits.open(f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/mosaic/nep_6p2arcsec_det{detector}_ch{channel}.fits')
fit_hdul = fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/mosaic/mosaic_det1_ch17.fits')
mosaic = fit_hdul[5].data
wcs = WCS(fit_hdul[5].header, naxis=2)


In [22]:
fit_hdul = fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/mosaic/mosaic_det1_ch4.fits')
mean = fit_hdul[1].data

In [25]:
# plot_map(mosaic-mean)

In [45]:
# high, low = np.nanpercentile(mosaic[mosaic>0], [1, 95])
# fig = plt.figure(figsize=(10, 10))
# ax = fig.add_subplot(111, projection=wcs)
# im = ax.imshow(mosaic, norm=LogNorm(vmin=low, vmax=high), origin='lower')

# ax.coords['ra'].set_axislabel('RA')
# ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
# ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
# ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
# ax.coords['dec'].set_axislabel('DEC')
# ax.grid(color='black', linestyle='--', alpha=0.5)

# cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
# cbar.set_label('MJy/sr')
# ax.set_title(f'Detector 1 Channel 16', fontsize=16)
# # plt.savefig('det1_ch16.png', dpi=400)


In [ ]:
# fig = plt.figure(figsize=(10, 10))

# # Calculate the center 20% of the mosaic
# ny, nx = mosaic.shape
# x_start, x_end = int(nx * 0.4), int(nx * 0.6)
# y_start, y_end = int(ny * 0.5), int(ny * 0.7)

# # Crop the mosaic and update the WCS
# mosaic_cropped = mosaic[y_start:y_end, x_start:x_end]
# wcs_cropped = wcs.slice((slice(y_start, y_end), slice(x_start, x_end)))

# ax = fig.add_subplot(111, projection=wcs_cropped)
# im = ax.imshow(mosaic_cropped, norm=LogNorm(vmin=low, vmax=high), origin='lower')

# ax.coords['ra'].set_axislabel('RA')
# ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
# ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
# ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
# ax.coords['dec'].set_axislabel('DEC')
# ax.grid(color='black', linestyle='--', alpha=0.5)

# cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
# cbar.set_label('MJy/sr')
# ax.set_title(f'Detector 2 Channel 5 (20% Zoom)', fontsize=16)



In [ ]:
detector = 1
channel = 12
cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/calibration/cal_det{detector}_ch{channel}.h5'

with h5py.File(cal_path, 'r') as f:
    O = f['O'][:]
    S = f['S'][:]
    D = f['D'][:]

skymap = S+np.mean(O)+np.mean(D[np.nonzero(D)])
skymap[S==0] = np.nan

### Make Gif

In [ ]:
display_range = []
for detector in tqdm([1,2,3,4,5,6]):
    channel = 7
    fit_hdul = fits.open(f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/mosaic/nep_6p2arcsec_det{detector}_ch{channel}.fits')
    # fit_hdul = fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/mosaic/nep_6p2arcsec_det4_ch12_20band.fits')
    mosaic = fit_hdul[0].data
    wcs = WCS(fit_hdul[0].header, naxis=2)
    display_range.append(np.nanpercentile(mosaic[mosaic>0], [0.01, 99]).tolist())

In [39]:
from PIL import Image

def make_gif(image_list, output_path, duration=500):
    """
    Create a GIF from a list of images.

    Parameters:
    - image_list: List of file paths to images.
    - output_path: Path to save the output GIF.
    - duration: Duration of each frame in milliseconds.
    """
    frames = [Image.open(img) for img in image_list]
    frames[0].save(output_path, save_all=True, append_images=frames[1:], duration=duration, loop=0)

In [ ]:
display_range[4] = [0.2, 0.6]
display_range[5] = [0.3, 0.7]

In [51]:
tbl = Table.read('/home/thomasli/spherex/spherex_nep_catalogues/spherex_channels.csv')
for detector in [1,]:
    for i, channel in enumerate(tqdm(range(1, 18))):
        fit_hdul = fits.open(f'/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/mosaic/mosaic_det{detector}_ch{channel}.fits')
        mosaic = fit_hdul[5].data
        wcs = WCS(fit_hdul[5].header, naxis=2)
        lam = tbl[tbl['band']==detector]['lmean'][i]
        # fig, ax = plot_map(mosaic, wcs=wcs, lowp=1, highp=95, colorbar=False)
        fig, ax = plot_map(mosaic, wcs=wcs, low=0.09, high=1, colorbar=True)
        ax.set_title(f'D{detector} Ch{channel} $\\lambda = {lam} \\mu m$')
        fig.savefig(f'/home/thomasli/spherex/selfcal/figures/mosaics/D{detector}_Ch{channel}.png', dpi=200, bbox_inches='tight')
        plt.close(fig)
    
    image_list = [f'/home/thomasli/spherex/selfcal/figures/mosaics/D{detector}_Ch{channel}.png' for channel in range(1,18)]
    
    output_path = f'/home/thomasli/spherex/selfcal/figures/D{detector}_mosaics.gif'
    make_gif(image_list, output_path)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [03:15<00:00, 11.51s/it]


In [30]:
image_list = [f'/home/thomasli/spherex/selfcal/figures/mosaics/D{detector}_Ch{channel}.png' for channel in range(1,10)]
output_path = f'/home/thomasli/spherex/selfcal/figures/D{detector}_mosaics.gif'
make_gif(image_list, output_path)

In [ ]:

# tbl = Table.read('/home/thomasli/spherex/spherex_nep_catalogues/spherex_channels.csv')
# for detector in [5,6]:
#     for i, channel in enumerate(tqdm(range(1, 18))):
#         fit_hdul = fits.open(f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/mosaic/nep_6p2arcsec_det{detector}_ch{channel}.fits')
#         mosaic = fit_hdul[0].data
#         wcs = WCS(fit_hdul[0].header, naxis=2)
#         lam = tbl[tbl['band']==detector]['lmean'][i]
        
#         fig = plt.figure(figsize=(10, 10))
#         ax = fig.add_subplot(111, projection=wcs)
#         high, low = display_range[detector-1]
#         im = ax.imshow(mosaic, norm=LogNorm(vmin=low, vmax=high), origin='lower')
    
#         ax.coords['ra'].set_axislabel('RA')
#         ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
#         ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
#         ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
#         ax.coords['dec'].set_axislabel('DEC')
#         ax.grid(color='black', linestyle='--', alpha=0.5)
    
#         cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
#         cbar.set_label('MJy/sr')
    
#         ax.set_title(f'D{detector} Ch{channel} $\\lambda = {lam} \\mu m$')
#         fig.savefig(f'/home/thomasli/spherex/selfcal/figures/mosaics/D{detector}_Ch{channel}.png', dpi=200, bbox_inches='tight')
#         plt.close(fig)
    
#     image_list = [f'/home/thomasli/spherex/selfcal/figures/mosaics/D{detector}_Ch{channel}.png' for channel in range(1,18)]
    
#     output_path = f'/home/thomasli/spherex/selfcal/figures/det{detector}_mosaics.gif'
#     make_gif(image_list, output_path)

In [ ]:


# Example usage:


### Plot O

In [ ]:
import warnings
import numpy as np
import pandas as pd # Make sure pandas is imported
from datetime import timedelta

# Astropy imports
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord, get_sun, EarthLocation
from astropy.utils.exceptions import AstropyWarning

def load_solar_flux_data(filepath):
    """
    Loads and prepares the solar flux data file for interpolation.
    This function is correct and remains unchanged.
    """
    col_names = [
        'fluxdate', 'fluxtime', 'fluxjulian', 'fluxcarrington',
        'fluxobsflux', 'fluxadjflux', 'fluxursi'
    ]
    
    try:
        df_flux = pd.read_csv(
            filepath,
            delim_whitespace=True,
            comment='#',
            header=None,
            skiprows=1,
            names=col_names
        )
        
        # Explicitly convert the columns we need to numeric data types.
        df_flux['fluxjulian'] = pd.to_numeric(df_flux['fluxjulian'], errors='coerce')
        df_flux['fluxobsflux'] = pd.to_numeric(df_flux['fluxobsflux'], errors='coerce')
        
        # Remove any rows where the conversion might have failed.
        df_flux.dropna(subset=['fluxjulian', 'fluxobsflux'], inplace=True)
        
        # Ensure the data is sorted by Julian Date for interpolation.
        df_flux.sort_values('fluxjulian', inplace=True)
        
        print(f"Successfully loaded and cleaned solar flux data from '{filepath}'.")
        return df_flux
    except FileNotFoundError:
        print(f"ERROR: Solar flux data file not found at '{filepath}'.")
        return None


def analyze_airglow_conditions(header, df_solar, satellite_altitude_km=700.0):
    """
    Calculates physical conditions relevant to He I 1.083um airglow intensity.

    This function now also returns the observation time in two formats:
    - A Python datetime object, perfect for time-series plots.
    - The Modified Julian Date (MJD) as a float.
    """
    # --- MODIFIED: Initialize the dictionary with new keys for date info ---
    results = {
        'obs_datetime': None,
        'mjd_avg': np.nan,
        'angle_to_sun_deg': np.nan,
        'angle_from_nadir_deg': np.nan,
        'f107_flux': np.nan,
    }

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", AstropyWarning)

        try:
            # --- Get Time and Pointing Info ---
            obs_time = Time(header['DATE-AVG'])
            
            # --- NEW: Add date information to the results dictionary ---
            # Add the Python datetime object for easy plotting with Matplotlib/Seaborn
            results['obs_datetime'] = obs_time.datetime
            # Add the MJD for numerical analysis or scatter plots vs. time
            results['mjd_avg'] = header.get('MJD-AVG', obs_time.mjd)

            pointing_ra = header['CRVAL1'] * u.deg
            pointing_dec = header['CRVAL2'] * u.deg
            pointing_coord = SkyCoord(ra=pointing_ra, dec=pointing_dec, frame='icrs')

            # --- Calculate Angle to Sun ---
            sun_coord = get_sun(obs_time)
            results['angle_to_sun_deg'] = pointing_coord.separation(sun_coord).to_value(u.deg)

            # --- Calculate Angle from Nadir ---
            sat_lat = header['HIERARCH SGT_LAT_MIDPT'] * u.deg
            sat_lon = header['HIERARCH SGT_LON_MIDPT'] * u.deg
            satellite_location = EarthLocation(
                lat=sat_lat, lon=sat_lon, height=satellite_altitude_km * u.km
            )
            satellite_itrs = satellite_location.get_itrs(obstime=obs_time)
            nadir_coord = SkyCoord(
                x=-satellite_itrs.x, y=-satellite_itrs.y, z=-satellite_itrs.z,
                frame='itrs', obstime=obs_time
            )
            results['angle_from_nadir_deg'] = pointing_coord.separation(nadir_coord).to_value(u.deg)

            # --- Interpolate Solar Flux ---
            obs_jd = obs_time.jd
            interpolated_flux = np.interp(
                obs_jd,
                df_solar['fluxjulian'].values,
                df_solar['fluxobsflux'].values
            )
            results['f107_flux'] = interpolated_flux

        except KeyError as e:
            print(f"Warning: Missing essential header keyword {e}. Some results will be NaN.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

    return results

In [ ]:
df_solar = load_solar_flux_data('misc/fluxtable.txt')
analyze_airglow_conditions(hdr, df_solar)

Successfully loaded and cleaned solar flux data from 'misc/fluxtable.txt'.


/tmp/ipykernel_2072135/3897997062.py:23: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_flux = pd.read_csv(


{'obs_datetime': datetime.datetime(2025, 5, 8, 22, 36, 12, 841000),
 'mjd_avg': 60803.941815286,
 'angle_to_sun_deg': np.float64(91.84231891860425),
 'angle_from_nadir_deg': np.float64(93.12615089783071),
 'f107_flux': np.float64(147.06221653148532)}

In [ ]:
detector = 1

reproj_list = glob.glob(f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/reprojected/*.h5')

result_list = []

for f in tqdm(reproj_list):
    exp_path = MakeMap.load_reproj_file(f, fields=['file_path'])['file_path'].decode('ascii')
    exp_hdul = fits.open(exp_path)
    hdr = exp_hdul[1].header
    result = analyze_airglow_conditions(hdr, df_solar)
    result_list.append(result)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3458/3458 [00:29<00:00, 116.83it/s]


In [ ]:
df = pd.DataFrame(result_list)

In [ ]:
cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/calibration/cal_det1_ch15-17_10band_newO.h5'

with h5py.File(cal_path, 'r') as f:
    O = f['O'][:]

In [ ]:
df

,obs_datetime,mjd_avg,angle_to_sun_deg,angle_from_nadir_deg,f107_flux
0,2025-06-18 08:41:27.928,60844.362129,92.958699,92.591968,140.123247
1,2025-07-02 08:40:49.831,60858.361688,92.714627,93.028258,127.229376
2,2025-05-19 03:18:19.503,60814.137726,93.166809,94.666292,117.547419
3,2025-05-21 17:08:56.715,60816.714545,95.914778,96.786369,118.110544
4,2025-05-04 07:49:41.043,60799.326169,88.888000,90.212364,153.993919
...,...,...,...,...,...
3453,2025-06-24 09:58:12.251,60850.415420,89.612813,89.656879,121.411222
3454,2025-06-19 02:40:00.393,60845.111116,95.860235,96.623979,136.028005
3455,2025-06-18 11:56:09.969,60844.497338,96.225987,96.027787,140.033108
3456,2025-05-26 09:43:56.508,60821.405515,89.474945,91.439899,133.017030


In [ ]:
# plt.scatter(df['obs_datetime'] , O.max(axis=1), s=1, edgecolor='None')

In [ ]:
# _ = plt.plot(np.arange(20), O[:, 146:166][0:100].T)

In [ ]:

chs = [5]
cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/calibration/cal_det{detector}_ch{chs[0]}.h5'

with h5py.File(cal_path, 'r') as f:
    O = f['O'][:]

In [ ]:
# plt.scatter(date_list, O, s=2, edgecolor='none')

# plt.ylim([0.14, 0.22])
# plt.xticks(rotation=45)
# plt.xlabel('Date')
# plt.ylabel('$F^n$ [MJy/sr]')
# plt.title()


In [ ]:
with h5py.File(cal_path, 'r') as f:
    D = f['D'][:]

In [ ]:
# plt.plot(wav_full[1][D[1:-1]!=0][1:-1], D[D!=0][1:-1])
# plt.ylabel('$F_p$ [MJy/sr]')
# plt.xlabel('Wavelength [um]')


### Plot Total Offset

In [ ]:
chs_list = [[1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17]]

mean_O = []
std_D = []
cal_factor = 0
for i, detector in enumerate([1, 2, 3, 4, 5, 6]):
    for j, chs in enumerate(chs_list):
        cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/calibration/cal_det{detector}_ch{chs[0]}.h5'

        if i > 1 and j == 0:
            prev_D = D_valid[-1]
            prev_O = mean_O[-1]

        with h5py.File(cal_path, 'r') as f:
            O = f['O'][:]
            D = f['D'][:]
        chunk_valid_mask = make_chunk_mask(chs, interp_factor=5)
        D_valid = D[chunk_valid_mask==1]

        if i > 1 and j == 0:
            cal_factor = D_valid[0] + np.mean(O) - prev_D - prev_O

        mean_O.append(np.mean(O)-cal_factor)
        std_D.append(np.std(D_valid))

In [ ]:
def interpolate_array(data_arr, interp_factor=5):
    interp_arr = np.hstack([
        np.linspace(data_arr[i], data_arr[i + 1], interp_factor, endpoint=False) 
        for i in range(len(data_arr) - 1)
    ] + [data_arr[-1]])  # Append the last element
    return interp_arr

wav_arr = np.array([])
det_edge = []
for band in tqdm([1, 2, 3, 4, 5, 6]):
    tbl = Table.read('/home/thomasli/spherex/spherex_nep_catalogues/spherex_channels.csv')
    sub_tbl = tbl[tbl['band'] == band]
    channel_edges = np.hstack([sub_tbl['lmin'].data, sub_tbl['lmax'].data[-1:]])
    fine_edges = interpolate_array(channel_edges, interp_factor=1)
    wav = ((fine_edges[0:17*1] + fine_edges[1:17*1 + 1])/2)
    wav_arr = np.concatenate((wav_arr, wav))
    det_edge.append(sub_tbl['lmin'].data[0])    
det_edge.append(sub_tbl['lmax'].data[-1])

In [ ]:
# plt.errorbar(wav_arr, mean_O, yerr=std_D, marker='.', ls='none', markersize=2, linewidth=1)
# for i in range(0, 7):
#     plt.axvline(x=det_edge[i], color='gray', linestyle='--', alpha=0.5, linewidth=0.5)
# # plt.grid()
# # add tick


In [ ]:
chs_list = [[1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17]]
tbl = Table.read('/home/thomasli/spherex/spherex_nep_catalogues/spherex_channels.csv')

D_full = []
wav_full = []
det_edge = []

for i, detector in enumerate([1, 2, 3, 4, 5, 6]):
    D_det = []
    wav_det = []
    for j, chs in enumerate(chs_list):
        cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/calibration/cal_det{detector}_ch{chs[0]}.h5'

        with h5py.File(cal_path, 'r') as f:
            O = f['O'][:]
            D = f['D'][:]
        chunk_valid_mask = make_chunk_mask(chs, interp_factor=5)

        D_valid = D[chunk_valid_mask==1]
        D_det.append(D_valid + np.mean(O))

    D_det = np.hstack(D_det)
    D_full.append(D_det)

    sub_tbl = tbl[tbl['band'] == detector]
    channel_edges = np.hstack([sub_tbl['lmin'].data, sub_tbl['lmax'].data[-1:]])
    fine_edges = interpolate_array(channel_edges, interp_factor=5)
    wav_det = ((fine_edges[0:17*5] + fine_edges[1:17*5 + 1])/2)
    wav_full.append(wav_det)

    det_edge.append(sub_tbl['lmin'].data[0])
det_edge.append(sub_tbl['lmax'].data[-1])


In [ ]:
cal_path = '/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/calibration/cal_det1_ch15-17_10band_100sig.h5'

with h5py.File(cal_path, 'r') as f:
    O = f['O'][:]
    S = f['S'][:]
    D = f['D'][:]

chunk_map = make_chunk_map(detector, interp_factor=10)
chunk_valid_mask = make_chunk_mask([15, 16, 17], interp_factor=10)
chunk_valid_mask[141:147] = 0
chunk_valid_mask[169:171] = 0

cal_D = D + np.mean(O)

In [ ]:
bin_D = np.array(((cal_D[::2] + cal_D[1::2])/2))[75: -3]

In [ ]:
D_full[0][75: -2] = bin_D

In [ ]:
# det_offset = [-0.028, -0.009, 0.014, 0.044, -0.039, -0.137]
# for i in range(len(D_full)):
#     plt.plot(wav_full[i], D_full[i],'o-', markersize=0.5, linewidth=0.5)

# for edge in det_edge:
#     plt.axvline(x=edge, color='gray', alpha=0.5, linewidth=0.5)

# plt.xlabel('Wavelength [um]')
# plt.ylabel('Offset [MJy/sr]')
# plt.yscale('log')
# # spec_line = [1.08, 2.12, 3.28, 1.875]
# # for line in spec_line:
# #     plt.axvline(x=line, color='red', linestyle='--', alpha=0.5, linewidth=0.5)
# # # plt.grid()

### PAH

In [ ]:
def extract_sky(cal_path):
    with h5py.File(cal_path, 'r') as f:
        O = f['O'][:]
        S = f['S'][:]
        D = f['D'][:]

    skymap = S+np.mean(O)+np.mean(D[np.nonzero(D)])
    skymap[S==0] = np.nan
    return skymap

In [ ]:
fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/mosaic/nep_6p2arcsec_det4_ch10_10band_nosource.fits').info()

Filename: /home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/mosaic/nep_6p2arcsec_det4_ch10_10band_nosource.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  MEAN_MAP      1 ImageHDU        24   (9171, 8806)   float32   
  2  MEAN_WEIGHT    1 ImageHDU        24   (9171, 8806)   float32   
  3  STD_MAP       1 ImageHDU        24   (9171, 8806)   float32   
  4  STD_WEIGHT    1 ImageHDU        24   (9171, 8806)   float32   
  5  SC_MEAN_MAP    1 ImageHDU        24   (9171, 8806)   float32   
  6  SC_MEAN_WEIGHT    1 ImageHDU        24   (9171, 8806)   float32   


In [ ]:
ch11_data = fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/mosaic/nep_6p2arcsec_det4_ch10_10band_nosource.fits')[5].data
ch13_data = fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/mosaic/nep_6p2arcsec_det4_ch14_10band_nosource.fits')[5].data
ch12_data = fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/mosaic/nep_6p2arcsec_det4_ch12_10band_nosource.fits')[5].data
wcs = WCS(fits.open('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/mosaic/nep_6p2arcsec_det4_ch14_10band_nosource.fits')[5].header, naxis=2)

# ch11_data = extract_sky('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/cal_det4_ch10.h5')
# ch12_data = extract_sky('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/nep_6p2arcsec_det4_ch12_20band.fits')
# ch13_data = extract_sky('/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/cal_det4_ch14.h5')
ch12_sim = (ch11_data + ch13_data)/2


In [ ]:
mpl.rcParams['figure.dpi'] = 200 # User can set this outside the class if needed

In [ ]:
# plot_map(ch12_data, wcs=wcs, lowp=10, highp=40)

In [ ]:
pah_map = ch12_data-ch12_sim
valid_mask = np.all([ch11_data!=0, ch12_data!=0, ch13_data!=0], axis=0)
pah_map[~valid_mask] = 0

In [ ]:
# fig = plt.figure(figsize=(10, 10))
# ax = fig.add_subplot(111, projection=wcs)
# im = ax.imshow(pah_map, norm=LogNorm(vmin=1e-3, vmax=5e-2), origin='lower')
# ax.set_title('PAH Map')

# ax.coords['ra'].set_axislabel('RA')
# ax.coords['ra'].set_axislabel_position('b')  # Ensure RA label is only at the bottom
# ax.coords['ra'].set_ticks_position('b')  # Set RA ticks only at the bottom
# ax.coords['ra'].set_ticklabel_position('b')  # Set RA tick labels only at the bottom
# ax.coords['dec'].set_axislabel('DEC')
# ax.grid(color='black', linestyle='--', alpha=0.5)

# cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.040, pad=0.04)
# cbar.set_label('MJy/sr')


In [ ]:
cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det4_6p2arcsec/calibration/nep_6p2arcsec_det4_ch12_20band.fits'
with h5py.File(cal_path, 'r') as f:
    D = f['D'][:]
    O = f['O'][:]
    S = f['S'][:]

### Plot D

In [ ]:
from py.SPHERExUtility import make_chunk_map, make_chunk_mask

detector = 1
chs_list = [[1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17]]
D_arr = np.array([])
chunk_map = make_chunk_map(detector, interp_factor=5)
for chs in chs_list:
    cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/calibration/cal_det{detector}_ch{chs[0]}.h5'
    chunk_valid_mask = make_chunk_mask(chs, interp_factor=5)

    with h5py.File(cal_path, 'r') as f:
        O = f['O'][:]
        S = f['S'][:]
        D_full = f['D'][:] + S.mean() + O.mean()
    D_offset = D_full[chunk_valid_mask==1]
    D_arr = np.concatenate((D_arr, D_offset))
    

In [ ]:
chs_list = [[1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17]]
D_full = []
for detector in [1,2,3]:
    D_det = []
    for chs in tqdm(chs_list):
        cal_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/calibration/cal_det{detector}_ch{chs[0]}.h5'
        mos_path = f'/home/thomasli/spherex/selfcal/outputs/nep_det{detector}_6p2arcsec/mosaic/nep_6p2arcsec_det{detector}_ch{chs[0]}.fits'
        chunk_valid_mask = make_chunk_mask(chs, interp_factor=5)
        mosaic = fits.open(mos_path)[0].data

        with h5py.File(cal_path, 'r') as f:
            O = f['O'][:]
            S = f['S'][:]
            D = f['D'][:]
        D_full.append(D[chunk_valid_mask==1] + O.mean())
    #     D_det += (D[1:-1] + mosaic[mosaic!=0].mean())
    # D_full = np.concatenate((D_full, D_det))

In [ ]:
D_full = np.hstack(D_full)

In [ ]:
# plt.plot(np.arange(len(D_full)), D_full, marker='.')

In [ ]:
# plt.plot(np.arange(1, 17*5+1), D_full,  linestyle='-', linewidth=0.5)

In [ ]:
def interpolate_array(data_arr, interp_factor=5):
    interp_arr = np.hstack([
        np.linspace(data_arr[i], data_arr[i + 1], interp_factor, endpoint=False) 
        for i in range(len(data_arr) - 1)
    ] + [data_arr[-1]])  # Append the last element
    return interp_arr

# band = 1
# tbl = Table.read('/home/thomasli/spherex/spherex_nep_catalogues/spherex_channels.csv')
# sub_tbl = tbl[tbl['band'] == band]
# channel_edges = np.hstack([sub_tbl['lmin'].data, sub_tbl['lmax'].data[-1:]])
# fine_edges = interpolate_array(channel_edges, interp_factor=5)

# wav = ((fine_edges[0:17*5] + fine_edges[1:17*5 + 1])/2)

wav_arr = np.array([])
for band in tqdm([1, 2, 3, 4, 5, 6]):
    tbl = Table.read('/home/thomasli/spherex/spherex_nep_catalogues/spherex_channels.csv')
    sub_tbl = tbl[tbl['band'] == band]
    channel_edges = np.hstack([sub_tbl['lmin'].data, sub_tbl['lmax'].data[-1:]])
    fine_edges = interpolate_array(channel_edges, interp_factor=5)

    wav = ((fine_edges[0:17*5] + fine_edges[1:17*5 + 1])/2)
    wav_arr = np.concatenate((wav_arr, wav))
    

In [ ]:
wav_arr

In [ ]:
# plt.scatter(wav_arr, D_full, s=1, edgecolors='none')
# plt.xlabel('Wavelength (um)')
# plt.ylabel('Detector Offset (MJy/sr)')
# for i in range(17):
#     plt.axvline(x=fine_edges[i*5], color='k', linestyle='--', linewidth=0.5)
# plt.axvline(x=fine_edges[i*5+5], color='k', linestyle='--', linewidth=0.5)
# plt.axvline(x=1.094, color='k', linestyle='--', linewidth=0.5)
# plt.axvline(x=2.12, color='k', linestyle='--', linewidth=0.5)


### Test Calibration

In [ ]:
reproj_list = sorted(glob.glob('outputs/nep_det1_6p2arcsec/reprojected/*.h5'))

det_idx_list = []
exp_idx_list = []
for file in tqdm(reproj_list):
    file_name = os.path.basename(file)
    det_idx_list.append(int(file_name[file_name.find('det_')+4:file_name.find('det_')+6]))
    exp_idx_list.append(int(file_name[file_name.find('exp_')+4:file_name.find('exp_')+8]))

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3458/3458 [00:00<00:00, 1218815.40it/s]


In [ ]:
cal_path = '/home/thomasli/spherex/selfcal/outputs/nep_det1_6p2arcsec/calibration/cal_det1_ch15-17_10band_newO.h5'

with h5py.File(cal_path, 'r') as f:
    O = f['O'][:]
    S = f['S'][:]
    # D = f['D'][:]
# D -= D.mean()
# O -= O.mean()

In [ ]:
detector = 1
chunk_map = make_chunk_map(detector, interp_factor=10)
chunk_valid_mask = make_chunk_mask([15, 16, 17], interp_factor=10)
chunk_valid_mask[141:146] = 0
chunk_valid_mask[166:] = 0

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:10<00:00, 15.74it/s]


In [ ]:
def interp_1d(arr, method='mp'):
    idx = np.arange(len(arr))
    start = np.where(arr[:-1] != arr[1:])[0]+1
    mean_idx = (start[:-1] + (start[1:] - 1))/2
    mean_val = arr[start[:-1]]

    arr[idx < start[0]] = mean_val[0]
    arr[idx >= start[-1]] = mean_val[-1]
    
    if method == 'mp':
        from mpsplines import MeanPreservingInterpolation as MPI
        # https://github.com/jararias/mpsplines
        mpi = MPI(yi=mean_val, xi=mean_idx)
        smooth_arr = mpi(idx)
    elif method == 'linear':
        smooth_arr = np.interp(idx, mean_idx, mean_val)
        
    return smooth_arr

def interp_2d_vertical(arr, method='mp'):
    return np.apply_along_axis(interp_1d, axis=0, arr=arr, method=method)

In [ ]:
# exp_idx = 1100
# cal = O[exp_idx]
# norm_cal = cal - np.min(cal[cal!=0])
# valid_mask = chunk_valid_mask[chunk_map]
# offset_map = norm_cal[chunk_map]
# offset_map_mp = interp_2d_vertical(offset_map, method='mp')
# offset_map_linear = interp_2d_vertical(offset_map, method='linear')

# reproj_file = MakeMap.load_reproj_file(reproj_list[exp_idx], fields=('file_path',))
# hdul = fits.open(reproj_file['file_path'].decode('ascii'))
# data = hdul[1].data

# fig, axes = plt.subplots(4, 1, figsize=(10, 12))
# axes[0].imshow(((data)*valid_mask)[0:400], norm=LogNorm(vmin=0.1, vmax=0.5))
# axes[0].set_title('Raw Exposure')
# axes[1].imshow(((data-offset_map)*valid_mask)[0:400], norm=LogNorm(vmin=0.1, vmax=0.5))
# axes[1].set_title('Calibrated (Discrete)')
# axes[2].imshow(((data-offset_map_mp)*valid_mask)[0:400], norm=LogNorm(vmin=0.1, vmax=0.5))
# axes[2].set_title('Calibrated (Mean-Preserving Interpolation)')
# axes[3].imshow(((data-offset_map_linear)*valid_mask)[0:400], norm=LogNorm(vmin=0.1, vmax=0.5))
# axes[3].set_title('Calibrated (Linear Interpolation)')


In [ ]:
reproj_list = sorted(glob.glob('outputs/nep_det1_6p2arcsec/reprojected/*.h5'))

det_idx_list = []
exp_idx_list = []
for file in tqdm(reproj_list):
    file_name = os.path.basename(file)
    det_idx_list.append(int(file_name[file_name.find('det_')+4:file_name.find('det_')+6]))
    exp_idx_list.append(int(file_name[file_name.find('exp_')+4:file_name.find('exp_')+8]))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3458/3458 [00:00<00:00, 957859.15it/s]


In [ ]:
reproj_sample = MakeMap.load_reproj_file(reproj_list[0], fields=['sub_data','grid_bitmask', 'grid_mapping'])
grid_mapping = reproj_sample['grid_mapping']

In [380]:
from SelfCal import MapHelper

In [384]:
importlib.reload(MapHelper)

<module 'SelfCal.MapHelper' from '/home/thomasli/spherex/selfcal/SelfCal/MapHelper.py'>

In [404]:
%%time
det_offset = MapHelper.compute_offset_map(O[0], chunk_map, interp_method='mp')


CPU times: user 1.2 s, sys: 137 μs, total: 1.2 s
Wall time: 1.19 s


In [406]:
%%time
# grid_offset = MapHelper.det_to_grid(grid_mapping, det_offset)
# sub_offset = MapHelper.bin2d(grid_offset, bin_factor=2)

CPU times: user 236 ms, sys: 174 ms, total: 409 ms
Wall time: 408 ms
